# Binomial Hypothesis Testing

We are comparing **simulation-based** and **exact binomial tests**.

---

## The Problem

Suppose we run `n` independent Bernoulli trials, each with probability of success `p0` under the null hypothesis (H0). We observe `k` successes.  
We want to test whether this observed result is consistent with H0.

- **Null hypothesis (H0):** true probability of success = `p0`.
- **Alternative hypotheses (H1):**
  - `less`: the true probability is smaller than `p0`.
  - `greater`: the true probability is larger than `p0`.
  - `two-sided`: the true probability is different from `p0`.

---

## Simulation Test

1. Simulate many repetitions (e.g. 10,000) of the experiment under H0.  
   Each repetition samples `n` trials with probability `p0`.  
2. Count how many of these simulated outcomes are at least as extreme as the observed `k`.  
3. The proportion of extreme outcomes is the **simulation p-value**.

This is an approximation: more simulations give more accuracy.

---

## Exact Binomial Test

The exact test does not simulate.  
It uses the **binomial probability mass function** directly:

$$
P(K = k) = \binom{n}{k} p_0^k (1-p_0)^{n-k}
$$

- For a one-sided test, the p-value is the probability of outcomes as extreme in one direction.  
- For a two-sided test, the p-value is the probability of outcomes as far (or farther) from the expected value `n * p0` as the observed `k`.  

This gives the **true p-value** under the null.

---

## Comparison

- The **simulation test** is useful when an exact test is not available or when teaching.  
- The **exact test** is precise and should be used when possible.  
- With enough simulations, both methods produce very similar results.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import binomtest

In [2]:
def simulation_binomial_test(observed_successes, n, p, alternative_hypothesis):
  #initialize null_outcomes
  null_outcomes = []
  
  #generate the simulated null distribution
  for i in range(10000):
    simulated_monthly_visitors = np.random.choice(['y', 'n'], size=n, p=[p, 1-p])
    num_purchased = np.sum(simulated_monthly_visitors == 'y')
    null_outcomes.append(num_purchased)

  null_outcomes = np.array(null_outcomes)

  if alternative_hypothesis == 'less':
    p_value = np.sum(null_outcomes <= observed_successes)/len(null_outcomes) 
  elif alternative_hypothesis == 'greater':
    p_value = np.sum(null_outcomes >= observed_successes)/len(null_outcomes)
  else:
    difference = np.abs(p*n - observed_successes)
    upper = p*n + difference
    lower = p*n - difference
    p_value = np.sum((null_outcomes >= upper) | (null_outcomes <= lower))/len(null_outcomes)
  
  #return the p-value
  return p_value

---
### Lower-Tail (One-Sided) Binomial Test

We test whether the observed number of successes is significantly **lower** than expected under the null hypothesis \(p = 0.1\).


In [3]:
print('lower tail one-sided test:')
p_value1 = simulation_binomial_test(45, 500, .1, alternative_hypothesis = 'less')
print("simulation p-value: ", p_value1)

p_value2 = binomtest(45, 500, .1, alternative = 'less').pvalue
print("binom_test p-value: ", p_value2)

lower tail one-sided test:
simulation p-value:  0.2531
binom_test p-value:  0.254689260562342


---
### Upper-Tail (One-Sided) Binomial Test

We test whether the observed number of successes is significantly **higher** than expected under the null hypothesis \(p = 0.1\).

In [4]:
print('upper tail one-sided test:')
p_value1 = simulation_binomial_test(53, 500, .1, alternative_hypothesis = 'greater')
print("simulation p-value: ", p_value1)

p_value2 = binomtest(53, 500, .1, alternative = 'greater').pvalue
print("binom_test p-value: ", p_value2)

upper tail one-sided test:
simulation p-value:  0.3426
binom_test p-value:  0.34838119949983054


---
### Two-Sided Binomial Test

We test whether the observed number of successes is **different** from what is expected under the null hypothesis \(p = 0.1\), regardless of direction.

In [5]:
print('two-sided test:')
p_value1 = simulation_binomial_test(42, 500, .1, alternative_hypothesis = 'not_equal')
print("simulation p-value: ", p_value1)

p_value2 = binomtest(42, 500, .1)
print("binom_test p-value: ", p_value2)

two-sided test:
simulation p-value:  0.2717
binom_test p-value:  BinomTestResult(k=42, n=500, alternative='two-sided', statistic=0.084, pvalue=0.2631383468731764)
